# Task 1: Term Deposit Subscription Prediction (Bank Marketing)
**DevelopersHub Corporation – Data Science Internship**

## Problem Statement
Predict whether a bank customer will subscribe to a term deposit as a result of a marketing campaign using the UCI Bank Marketing Dataset.

## Objective
- Build classification models (Logistic Regression, Random Forest)
- Evaluate using Confusion Matrix, F1-Score, and ROC Curve
- Explain predictions using SHAP

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, roc_curve, auc, ConfusionMatrixDisplay
)
import shap

print('All libraries imported successfully!')

## 2. Load Dataset
> Download from: https://archive.ics.uci.edu/ml/datasets/bank+marketing  
> File: `bank-full.csv` (semicolon-separated)

In [ ]:
# Load dataset
df = pd.read_csv('bank-full.csv', sep=';')
print('Shape:', df.shape)
df.head()

## 3. Dataset Description & Exploration

In [ ]:
# Basic info
print('\n--- Dataset Info ---')
df.info()

print('\n--- Missing Values ---')
print(df.isnull().sum())

print('\n--- Target Variable Distribution ---')
print(df['y'].value_counts())

In [ ]:
# Statistical summary
df.describe()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution
plt.figure(figsize=(6, 4))
df['y'].value_counts().plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black')
plt.title('Term Deposit Subscription Distribution')
plt.xlabel('Subscribed (yes/no)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150)
plt.show()

In [ ]:
# Age distribution by subscription
plt.figure(figsize=(10, 4))
df[df['y']=='yes']['age'].hist(alpha=0.7, label='Subscribed', bins=30, color='green')
df[df['y']=='no']['age'].hist(alpha=0.7, label='Not Subscribed', bins=30, color='red')
plt.legend()
plt.title('Age Distribution by Subscription')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150)
plt.show()

In [ ]:
# Subscription rate by job type
job_sub = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()).sort_values(ascending=False)
plt.figure(figsize=(12, 5))
job_sub.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Subscription Rate by Job Type')
plt.ylabel('Subscription Rate')
plt.xlabel('Job')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('job_subscription.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(10, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

## 5. Data Preprocessing & Feature Encoding

In [ ]:
# Copy dataframe
df_enc = df.copy()

# Encode target variable
df_enc['y'] = df_enc['y'].map({'yes': 1, 'no': 0})

# Identify categorical columns
cat_cols = df_enc.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)

# Label Encoding for all categorical features
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

print('\nEncoding complete!')
df_enc.head()

In [ ]:
# Split features and target
X = df_enc.drop('y', axis=1)
y = df_enc['y']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

## 6. Model Building & Training

In [ ]:
# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_sc, y_train)
lr_pred = lr_model.predict(X_test_sc)
lr_prob = lr_model.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, lr_pred))
print(f'F1 Score: {f1_score(y_test, lr_pred):.4f}')

In [ ]:
# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, rf_pred))
print(f'F1 Score: {f1_score(y_test, rf_pred):.4f}')

## 7. Model Evaluation

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pred, title in zip(axes,
                            [lr_pred, rf_pred],
                            ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'{title} Confusion Matrix')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))

for prob, label, color in zip(
    [lr_prob, rf_prob],
    ['Logistic Regression', 'Random Forest'],
    ['blue', 'green']
):
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{label} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.head(10).plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 8. Explainable AI with SHAP

In [ ]:
# SHAP Explainer for Random Forest
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# Global Feature Importance (Summary Plot)
plt.figure()
shap.summary_plot(shap_values[1], X_test, feature_names=X.columns.tolist(),
                  plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance')
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Beeswarm Summary Plot
plt.figure()
shap.summary_plot(shap_values[1], X_test, feature_names=X.columns.tolist(), show=False)
plt.title('SHAP Summary Plot (Beeswarm)')
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Explain 5 individual predictions using SHAP Waterfall plots
print('=== Explaining 5 Individual Predictions ===')

# Convert to SHAP Explanation object
shap_explanation = shap.Explanation(
    values=shap_values[1],
    base_values=explainer.expected_value[1],
    data=X_test.values,
    feature_names=X.columns.tolist()
)

for i in range(5):
    plt.figure()
    shap.waterfall_plot(shap_explanation[i], show=False)
    actual = y_test.iloc[i]
    predicted = rf_pred[i]
    plt.title(f'Prediction #{i+1} | Actual: {"Yes" if actual==1 else "No"} | Predicted: {"Yes" if predicted==1 else "No"}')
    plt.tight_layout()
    plt.savefig(f'shap_prediction_{i+1}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Prediction {i+1}: Actual={actual}, Predicted={predicted}')

## 9. Final Conclusion & Insights

### Key Findings:

1. **Dataset**: The Bank Marketing dataset contains ~45,000 records with 16 features. The target is highly imbalanced (~88% 'no', ~12% 'yes').

2. **Model Performance**:
   - **Logistic Regression**: Provides a solid baseline. Works well after feature scaling.
   - **Random Forest**: Outperforms Logistic Regression with a higher F1-Score and AUC, capturing complex non-linear patterns.

3. **Important Features** (from SHAP & Feature Importance):
   - `duration` (last contact duration): Most influential — longer calls → higher subscription
   - `poutcome` (previous campaign outcome): Positive previous outcomes strongly predict subscription
   - `balance`: Higher account balance → more likely to subscribe
   - `month`, `age`, `job` also play significant roles

4. **SHAP Insights**: Individual SHAP waterfall plots reveal that high call duration and positive previous outcomes push predictions toward 'yes', while 'unknown' job types or negative prior campaigns push toward 'no'.

5. **Business Recommendation**: Focus marketing calls on customers with:
   - Positive previous campaign outcomes
   - Higher account balances
   - Retired or student job categories (higher subscription rates)